In [1]:
# Preparing pathing
%load_ext autoreload
%autoreload 2
from titanic_ml import paths
import matplotlib.pyplot as plt
import pandas as pd
from titanic_ml.common.data.eda import summarize_categorical_column, summarize_numerical_column
from titanic_ml.common.data.eda import run_eda 


from ast import For
from cmath import exp

from titanic_ml.common.experiments.runner import run_experiments, run_experiment_group_workflow
from titanic_ml.common.experiments.config import ALL_EXPERIMENTS
from titanic_ml.common.experiments.report import experiment_report, experiment_group_summary_report, baseline_summary_to_markdown, workflow_report
from titanic_ml.common.experiments.save_load import save_results, load_results, save_configs, load_configs
from titanic_ml.common.experiments.compare import leaderboard, compare_experiment_groups, summarize_group_comparison, titanic_notes_leaderboard, model_progression
from titanic_ml.common.models.registry import MODEL_REGISTRY
from titanic_ml.feature_engineering import add_family_features, add_has_cabin, add_title, add_full_title_feature



In [2]:
TARGET = "Survived"
ALL_EXPERIMENTS = ALL_EXPERIMENTS
for experiment_name, exp_config in ALL_EXPERIMENTS.items():
    print(f"Experiment: {experiment_name}")

train_df = pd.read_csv(paths.TRAIN_PATH)

exp_configs = ALL_EXPERIMENTS["fe07__age_imputation_title_pclass"]

# Line to rerun all experiments to update the results with the latest code changes. This will take a while.
# Uncomment to run all experiments and update results.
# for Name, exp_config in ALL_EXPERIMENTS.items():
#     print(f"Running {Name} experiments...")
#     exp_result = run_experiments(train_df, exp_config, target=TARGET, verbose=True, debug=True)
#     save_results(exp_result)
#     save_configs(exp_config)



Experiment: baseline__raw
Experiment: fe01__family
Experiment: fe02__has_cabin
Experiment: fe03__deck
Experiment: fe04__cabin_features
Experiment: fe05__title
Experiment: fe06__age_imputation_title
Experiment: fe07__age_imputation_title_pclass


In [3]:
# print("Experiment Configurations:")
# print(exp_configs)
# for exp_config in exp_configs:
#     print(exp_config)

In [4]:
# Work flow for running an experiment group, comparing it to the baseline, and generating a report. 
# This is the main workflow for analyzing the results of an experiment group and generating insights from it.
workflow = run_experiment_group_workflow(
    df=train_df,
    experiment_configs=exp_configs,
    target=TARGET,
)

print("Workflow completed. Here are the results:")
print("Comparison between baseline and feature engineering group:")
print(workflow["comparison"])
print("Summary of comparison:")
print(workflow["summary"])
print("Leaderboard:")
print(workflow["leaderboard"])

running exp: {'name': 'fe07__age_imputation_title_pclass__logreg', 'features': ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked'], 'feature_engineering': [<function age_imputed_by_title_pclass at 0x000001664A2F8790>], 'preprocessing': {'numeric_features': ['Age', 'SibSp', 'Parch', 'Fare'], 'onehot_features': ['Sex', 'Embarked'], 'ordinal_features': ['Pclass'], 'numeric_imputer': 'median', 'categorical_imputer': 'most_frequent', 'scaler': 'standard'}, 'model_name': 'logreg', 'model_params': {'max_iter': 1000, 'random_state': 42}, 'evaluation': {'method': 'cross_validation', 'cv': 5, 'scoring': ['accuracy', 'precision', 'recall', 'f1'], 'return_train_score': True, 'n_jobs': -1}, 'notes': 'Feature engineering 06: imputing age by grouped impute using title and pclass, expected to give better results then imputing with only title', 'stage': 'fe07', 'feature_group': 'age_imputation_title_pclass', 'group': 'fe07__age_imputation_title_pclass'}
running exp: {'name': 'fe07__age_imput

In [5]:
# Generate a full report for the workflow, including the comparison, summary, and leaderboard. 
# This will be a markdown report that can be easily shared and visualized.
# Mainly used for generating the report for the notebook, but can also be used for generating reports for individual experiment groups or comparisons.
full_report = workflow_report(workflow)

print("Full workflow report:")
print()
print('Report')
print(full_report['report'])
print()
print('Leaderboard')
print(full_report['leaderboard'])

Full workflow report:

Report
### fe07__age_imputation_title_pclass

<details>
<summary>Experiment details</summary>

_Description pending._

<details>
<summary>Comparison details</summary>

#### Comparison vs baseline__raw

| reference_group   | compare_group                     | model_name    |   test_accuracy_mean_reference |   test_accuracy_mean_compare |   test_accuracy_mean_delta |   test_f1_mean_reference |   test_f1_mean_compare |   test_f1_mean_delta |
|:------------------|:----------------------------------|:--------------|-------------------------------:|-----------------------------:|---------------------------:|-------------------------:|-----------------------:|---------------------:|
| baseline__raw     | fe07__age_imputation_title_pclass | logreg        |                          0.786 |                        0.799 |                      0.013 |                    0.713 |                  0.725 |                0.012 |
| baseline__raw     | fe07__age_imputation_title_

In [6]:
# Reminder of how to run a single experiment if needed. 
# This is useful for when we want to generate reports or comparisons without rerunning all experiments.

# exp_config = [exp_config]
# exp_result = run_experiments(train_df, exp_config, target=TARGET,)
# exp_report = experiment_report(exp_result, exp_config, print_report=True)
# save_results(exp_result)
# save_configs(exp_config)

In [7]:
# Summary for baseline experiment to use in report, 
# since we can't compare it to itself.

# baseline_summary = baseline_summary_to_markdown(exp_result)
# print("Baseline summary:")
# print(baseline_summary)

In [8]:
# Reminder for how to load results and configs if needed. 
# This is useful for when we want to generate reports or comparisons without rerunning all experiments.

# all_results = load_results()
# print("Loaded results:")
# print(all_results)

# all_configs = load_configs()
# print("Loaded configs:")
# print(all_configs)

In [9]:
# print(workflow["all_results"])

In [10]:
for model in MODEL_REGISTRY:
    model_progression_df = model_progression(workflow["all_results"], model_name=model, metric="test_accuracy_mean")
    print(f"Model progression for {model}:")
    print(model_progression_df)
    print()

Model progression for logreg:
      stage                feature_group  test_accuracy_mean
0  baseline                          raw               0.786
1      fe01                       family               0.795
2      fe02                    has_cabin               0.791
3      fe03                         deck               0.791
4      fe04               cabin_features               0.791
5      fe05                        title               0.825
6      fe06         age_imputation_title               0.787
7      fe07  age_imputation_title_pclass               0.799

Model progression for knn:
      stage                feature_group  test_accuracy_mean
0  baseline                          raw               0.809
1      fe01                       family               0.805
2      fe02                    has_cabin               0.809
3      fe03                         deck               0.818
4      fe04               cabin_features               0.817
5      fe05                

In [11]:
exp_df = train_df
exp_df = add_family_features(train_df)

print(exp_df)

     PassengerId  Survived  Pclass  \
0              1         0       3   
1              2         1       1   
2              3         1       3   
3              4         1       1   
4              5         0       3   
..           ...       ...     ...   
886          887         0       2   
887          888         1       1   
888          889         0       3   
889          890         1       1   
890          891         0       3   

                                                  Name     Sex   Age  SibSp  \
0                              Braund, Mr. Owen Harris    male  22.0      1   
1    Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                               Heikkinen, Miss. Laina  female  26.0      0   
3         Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                             Allen, Mr. William Henry    male  35.0      0   
..                                                 ...     ...   ... 

In [12]:
exp_df[['Fare','FamilySize', 'Pclass']]

,Fare,FamilySize,Pclass
0,7.2500,2,3
1,71.2833,2,1
2,7.9250,1,3
3,53.1000,2,1
4,8.0500,1,3
...,...,...,...
886,13.0000,1,2
887,30.0000,1,1
888,23.4500,4,3
889,30.0000,1,1


In [13]:
exp_df['Fare/Person'] = exp_df['Fare']/exp_df['FamilySize']
print(exp_df['Fare/Person'])

0       3.62500
1      35.64165
2       7.92500
3      26.55000
4       8.05000
         ...   
886    13.00000
887    30.00000
888     5.86250
889    30.00000
890     7.75000
Name: Fare/Person, Length: 891, dtype: float64


In [14]:
bins = [0, 14, 35, 60, 100]
labels = ['0', '2', '3', '1']
exp_df['Age_bin'] = pd.cut(exp_df['Age'], bins=bins, labels=labels, right=False)

In [20]:
for age in range(0,18):
    test_df = exp_df[exp_df['Age'] == age][['Age','Survived']]
    test_df = test_df.groupby('Survived').value_counts()
    print(test_df)

Series([], Name: count, dtype: int64)
Survived  Age
0         1.0    2
1         1.0    5
Name: count, dtype: int64
Survived  Age
0         2.0    7
1         2.0    3
Name: count, dtype: int64
Survived  Age
0         3.0    1
1         3.0    5
Name: count, dtype: int64
Survived  Age
0         4.0    3
1         4.0    7
Name: count, dtype: int64
Survived  Age
1         5.0    4
Name: count, dtype: int64
Survived  Age
0         6.0    1
1         6.0    2
Name: count, dtype: int64
Survived  Age
0         7.0    2
1         7.0    1
Name: count, dtype: int64
Survived  Age
0         8.0    2
1         8.0    2
Name: count, dtype: int64
Survived  Age
0         9.0    6
1         9.0    2
Name: count, dtype: int64
Survived  Age 
0         10.0    2
Name: count, dtype: int64
Survived  Age 
0         11.0    3
1         11.0    1
Name: count, dtype: int64
Survived  Age 
1         12.0    1
Name: count, dtype: int64
Survived  Age 
1         13.0    2
Name: count, dtype: int64
Survived  Age 


In [ ]:
exp_df = add_family_features(train_df)
test_df = exp_df[['FamilySize','Fare', 'Pclass']]
uniques = test_df['Fare'].unique()
for fare in uniques:
    print(test_df[test_df['Fare']==fare].groupby('Fare').value_counts())

Fare  FamilySize  Pclass
7.25  1           3         12
      2           3          1
Name: count, dtype: int64
Fare     FamilySize  Pclass
71.2833  2           1         1
Name: count, dtype: int64
Fare   FamilySize  Pclass
7.925  1           3         13
       2           3          2
       3           3          2
       7           3          1
Name: count, dtype: int64
Fare  FamilySize  Pclass
53.1  2           1         5
Name: count, dtype: int64
Fare  FamilySize  Pclass
8.05  1           3         43
Name: count, dtype: int64
Fare    FamilySize  Pclass
8.4583  1           3         1
Name: count, dtype: int64
Fare     FamilySize  Pclass
51.8625  1           1         1
         2           1         1
Name: count, dtype: int64
Fare    FamilySize  Pclass
21.075  5           3         4
Name: count, dtype: int64
Fare     FamilySize  Pclass
11.1333  3           3         3
Name: count, dtype: int64
Fare     FamilySize  Pclass
30.0708  2           2         2
Name: count, dtype:

In [25]:
test_df = train_df[['Ticket','Fare', 'Pclass']]
uniques = test_df['Ticket'].unique()
for ticket in uniques:
    print(test_df[test_df['Ticket']==ticket].groupby('Ticket').value_counts())

Ticket     Fare  Pclass
A/5 21171  7.25  3         1
Name: count, dtype: int64
Ticket    Fare     Pclass
PC 17599  71.2833  1         1
Name: count, dtype: int64
Ticket            Fare   Pclass
STON/O2. 3101282  7.925  3         1
Name: count, dtype: int64
Ticket  Fare  Pclass
113803  53.1  1         2
Name: count, dtype: int64
Ticket  Fare  Pclass
373450  8.05  3         1
Name: count, dtype: int64
Ticket  Fare    Pclass
330877  8.4583  3         1
Name: count, dtype: int64
Ticket  Fare     Pclass
17463   51.8625  1         1
Name: count, dtype: int64
Ticket  Fare    Pclass
349909  21.075  3         4
Name: count, dtype: int64
Ticket  Fare     Pclass
347742  11.1333  3         3
Name: count, dtype: int64
Ticket  Fare     Pclass
237736  30.0708  2         2
Name: count, dtype: int64
Ticket   Fare  Pclass
PP 9549  16.7  3         2
Name: count, dtype: int64
Ticket  Fare   Pclass
113783  26.55  1         1
Name: count, dtype: int64
Ticket     Fare  Pclass
A/5. 2151  8.05  3         1
Nam